# CS3081 — Assignment: Uncertainty and Bayesian Networks
**Bayan Abduldaym**  |  **Student ID: 24109196**

Exercise 5(c): Python with pgmpy

In [1]:
%pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

model = DiscreteBayesianNetwork([
    ("heatwave", "overload"), ("heatwave", "powercut"),
    ("overload", "powercut"), ("powercut", "cancel")
])

heatwave = TabularCPD("heatwave", 2, [[0.3], [0.7]],
                      state_names={"heatwave": ["yes", "no"]})
overload = TabularCPD("overload", 2,
                      [[0.6, 0.1], [0.4, 0.9]],
                      evidence=["heatwave"], evidence_card=[2],
                      state_names={"overload": ["yes", "no"],
                                   "heatwave": ["yes", "no"]})
powercut = TabularCPD("powercut", 2,
                      [[0.7, 0.2, 0.5, 0.05],
                       [0.3, 0.8, 0.5, 0.95]],
                      evidence=["heatwave", "overload"], evidence_card=[2, 2],
                      state_names={"powercut": ["yes", "no"],
                                   "heatwave": ["yes", "no"],
                                   "overload": ["yes", "no"]})
cancel = TabularCPD("cancel", 2, [[0.8, 0.1], [0.2, 0.9]],
                    evidence=["powercut"], evidence_card=[2],
                    state_names={"cancel": ["yes", "no"],
                                 "powercut": ["yes", "no"]})
model.add_cpds(heatwave, overload, powercut, cancel)
print("Model correct:", model.check_model())

Model correct: True


In [3]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)
result = inference.query(variables=["heatwave"], evidence={"cancel": "yes"}, show_progress=False)
print(result)
exact = result.get_value(heatwave="yes")
print(f"Exact P(H=yes | L=yes): {exact:.4f}")

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+
Exact P(H=yes | L=yes): 0.5367


In [4]:
from pgmpy.sampling import BayesianModelSampling
from pgmpy.factors.discrete import State

samples = BayesianModelSampling(model).rejection_sample(
    evidence=[State("cancel", "yes")], size=10000, show_progress=False)
estimate = (samples["heatwave"] == "yes").mean()
print(f"Rejection sampling estimate (10000 samples): {estimate:.4f}")
print(f"Exact answer: {exact:.4f}")

Rejection sampling estimate (10000 samples): 0.5386
Exact answer: 0.5367


The exact answer is **0.5367**, while rejection sampling gives an estimate that may be slightly different. The estimate changes because of randomness, and using many samples usually makes it closer to the exact answer.